# Module 07: Axes Customization
**CHIRPS Rainfall Data – Ethiopia**

Fine-tune every aspect of your plot axes: limits, ticks, scales, secondary axes,
aspect ratios, and spines.



In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator, AutoLocator, ScalarFormatter, FuncFormatter

ds = xr.open_dataset(r'../chirps_1981_2022.nc', engine='netcdf4')
ts = ds.precip.isel(latitude=180, longitude=174)
ts_df = pd.DataFrame({'precip': ts.values}, index=pd.DatetimeIndex(ts.time.values))
monthly_mean = ts_df.groupby(ts_df.index.month)['precip'].mean()
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print("Data loaded")



---
## 7.1 Axis Limits (`set_xlim` / `set_ylim`)

Control which portion of the data is visible. Useful for zooming, excluding outliers,
or ensuring consistent scales across subplots.



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Full range
axes[0].plot(ts_df.index, ts_df['precip'], linewidth=0.5, color='steelblue')
axes[0].set_title('Full range (default limits)')
axes[0].set_ylabel('Precip (mm/month)')

# Zoom to 1990s
axes[1].plot(ts_df.index, ts_df['precip'], linewidth=0.5, color='steelblue')
axes[1].set_xlim(pd.Timestamp('1990-01-01'), pd.Timestamp('1999-12-01'))
axes[1].set_title('Zoom: 1990–1999')

# Zoom and clip y-axis
axes[2].plot(ts_df.index, ts_df['precip'], linewidth=0.5, color='steelblue')
axes[2].set_xlim(pd.Timestamp('1990-01-01'), pd.Timestamp('1999-12-01'))
axes[2].set_ylim(0, 200)
axes[2].set_title('Zoom: 1990–1999, y=[0,200]')

for ax in axes:
    ax.set_xlabel('Date')
plt.tight_layout(); plt.show()



---
## 7.2 Tick Locators and Formatters

**Locators** control where ticks are placed (`MultipleLocator`, `AutoLocator`, `MaxNLocator`, etc.).
**Formatters** control the label text (`ScalarFormatter`, `FuncFormatter`, `DateFormatter`).



In [ ]:
from matplotlib.ticker import MultipleLocator, AutoLocator, FormatStrFormatter

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Default
axes[0,0].plot(months, monthly_mean, 'o-', color='steelblue')
axes[0,0].set_title('Default ticks')

# MultipleLocator
axes[0,1].plot(months, monthly_mean, 'o-', color='steelblue')
axes[0,1].yaxis.set_major_locator(MultipleLocator(50))
axes[0,1].yaxis.set_minor_locator(MultipleLocator(10))
axes[0,1].xaxis.set_major_locator(MultipleLocator(2))
axes[0,1].set_title('MultipleLocator(50/10/2)')
axes[0,1].grid(True, which='both', alpha=0.3)

# FormatStrFormatter
axes[1,0].plot(months, monthly_mean, 'o-', color='steelblue')
axes[1,0].yaxis.set_major_formatter(FormatStrFormatter('%.0f mm'))
axes[1,0].set_title('FormatStrFormatter')

# FuncFormatter
def rain_category(x, pos):
    if x < 50: return f'{x:.0f} (dry)'
    if x < 150: return f'{x:.0f} (mod.)'
    return f'{x:.0f} (wet)'

axes[1,1].plot(months, monthly_mean, 'o-', color='steelblue')
axes[1,1].yaxis.set_major_formatter(FuncFormatter(rain_category))
axes[1,1].set_title('FuncFormatter – custom labels')
axes[1,1].tick_params(axis='x', rotation=45)

for ax in axes.flat:
    ax.set_ylabel('Precip (mm/month)')
plt.tight_layout(); plt.show()



In [ ]:
# DateFormatter for time axis
from matplotlib.dates import DateFormatter, MonthLocator, YearLocator

fig, ax = plt.subplots(figsize=(14, 4))
sub = ts_df.loc['2005':'2007']
ax.plot(sub.index, sub['precip'], 'o-', color='darkred', markersize=4)

ax.xaxis.set_major_locator(YearLocator())
ax.xaxis.set_major_formatter(DateFormatter('%Y'))
ax.xaxis.set_minor_locator(MonthLocator())
ax.xaxis.set_minor_formatter(DateFormatter('%b'))

ax.set_title('Date formatting – YearLocator + MonthLocator')
ax.set_ylabel('Precipitation (mm/month)')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.show()



---
## 7.3 Logarithmic Scales

Log scales are useful when data spans multiple orders of magnitude.
Use `semilogy`, `semilogx`, or `loglog` (or `ax.set_yscale('log')`).



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Linear
axes[0].plot(months, monthly_mean, 'o-', color='steelblue')
axes[0].set_title('Linear scale')
axes[0].set_ylabel('Precip (mm/month)')

# Log y
axes[1].plot(months, monthly_mean, 'o-', color='steelblue')
axes[1].set_yscale('log')
axes[1].set_title('Log y scale')
axes[1].set_ylabel('Precip (mm) [log]')
axes[1].grid(True, which='both', alpha=0.3)

# Symlog (handles zero)
symlog_data = monthly_mean.copy()
axes[2].plot(months, symlog_data, 'o-', color='steelblue')
axes[2].set_yscale('symlog', linthresh=10)
axes[2].set_title('Symlog scale (linthresh=10)')
axes[2].set_ylabel('Precip (mm) [symlog]')
axes[2].grid(True, which='both', alpha=0.3)

for ax in axes:
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()



---
## 7.4 Secondary Axes

Secondary axes share the same data space but have different tick labels and scales.
Useful for showing unit conversions or complementary variables.



In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

# Primary axis – monthly climatology (mm)
ax1.plot(months, monthly_mean, 'o-', color='steelblue', linewidth=2, markersize=6)
ax1.set_xlabel('Month'); ax1.set_ylabel('Precipitation (mm/month)', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

# Secondary axis – same data in metres
ax2 = ax1.twinx()
ax2.plot(months, monthly_mean / 1000, 's--', color='darkorange', linewidth=1.5, markersize=5)
ax2.set_ylabel('Precipitation (m/month)', color='darkorange')
ax2.tick_params(axis='y', labelcolor='darkorange')

# Secondary x-axis – month numbers
ax3 = ax1.secondary_xaxis('top')
ax3.set_xlabel('Month number')
ax3.set_xticks(range(12))
ax3.set_xticklabels(range(1, 13))

ax1.set_title('Dual-axis: mm (left) and m (right)')
plt.tight_layout(); plt.show()



In [ ]:
# Secondary axis with a conversion: mm to inches
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.bar(months, monthly_mean, color='steelblue', alpha=0.7)
ax1.set_xlabel('Month'); ax1.set_ylabel('Precipitation (mm/month)')
ax1.tick_params(axis='x', rotation=45)

def mm_to_inches(mm):
    return mm / 25.4

def inches_to_mm(inches):
    return inches * 25.4

ax2 = ax1.secondary_yaxis('right', functions=(mm_to_inches, inches_to_mm))
ax2.set_ylabel('Precipitation (inches/month)')

ax1.set_title('Dual units: mm and inches')
plt.tight_layout(); plt.show()



---
## 7.5 Aspect Ratio

`ax.set_aspect()` controls the physical aspect ratio of the data coordinates.
`'equal'` ensures one data unit is the same length on both axes.
`'auto'` lets Matplotlib fill the axes box.
A numeric value sets `y / x`.



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# auto
axes[0].plot(months, monthly_mean, 'o-', color='steelblue')
axes[0].set_aspect('auto')
axes[0].set_title('aspect="auto"')

# equal (distorts here because ranges differ)
axes[1].plot(months, monthly_mean, 'o-', color='steelblue')
axes[1].set_aspect('equal')
axes[1].set_title('aspect="equal"')

# numeric: y-unit = 2 x-unit
axes[2].plot(months, monthly_mean, 'o-', color='steelblue')
axes[2].set_aspect(0.5)
axes[2].set_title('aspect=0.5')

for ax in axes:
    ax.set_ylabel('Precip (mm/month)')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()



In [ ]:
# Aspect ratio on a scatter plot with CHIRPS spatial slice
lat_slice = ds.precip.isel(time=0, longitude=slice(100, 200))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wrong aspect – distorted map
im1 = axes[0].pcolormesh(ds.longitude[100:200], ds.latitude, lat_slice,
                          cmap='Blues', shading='auto')
axes[0].set_title('Distorted (default aspect)')
plt.colorbar(im1, ax=axes[0], label='mm')

# Correct aspect – lat/lon ≈ 1 at equator
im2 = axes[1].pcolormesh(ds.longitude[100:200], ds.latitude, lat_slice,
                          cmap='Blues', shading='auto')
axes[1].set_aspect(1.0 / np.cos(np.deg2rad(9)))  # ≈1.01 near equator
axes[1].set_title('Corrected aspect (cos(lat))')
plt.colorbar(im2, ax=axes[1], label='mm')

for ax in axes:
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()



---
## 7.6 Spines and Borders

Spines are the lines connecting the axes ticks. You can move, hide, or style them
individually for creative effects.



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Default spines
axes[0,0].plot(months, monthly_mean, 'o-', color='steelblue')
axes[0,0].set_title('Default spines')

# Hide top and right
axes[0,1].plot(months, monthly_mean, 'o-', color='steelblue')
axes[0,1].spines['top'].set_visible(False)
axes[0,1].spines['right'].set_visible(False)
axes[0,1].set_title('Hide top & right')

# Coloured spines
axes[1,0].plot(months, monthly_mean, 'o-', color='darkgreen', linewidth=2)
axes[1,0].spines['left'].set_color('darkgreen')
axes[1,0].spines['left'].set_linewidth(2)
axes[1,0].spines['bottom'].set_color('darkgreen')
axes[1,0].spines['bottom'].set_linewidth(2)
axes[1,0].spines['top'].set_visible(False)
axes[1,0].spines['right'].set_visible(False)
axes[1,0].set_title('Coloured spines')

# Offset spine
axes[1,1].plot(months, monthly_mean, 'o-', color='steelblue')
axes[1,1].spines['left'].set_position(('outward', 10))
axes[1,1].spines['bottom'].set_position(('outward', 10))
axes[1,1].spines['top'].set_visible(False)
axes[1,1].spines['right'].set_visible(False)
axes[1,1].set_title('Offset spines (10 pts)')

for ax in axes.flat:
    ax.set_ylabel('Precip (mm/month)')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()



In [ ]:
# Advanced spine customisation – "Matplotlib styled" plot
fig, ax = plt.subplots(figsize=(10, 5))

ax.fill_between(months, monthly_mean, alpha=0.3, color='coral')
ax.plot(months, monthly_mean, 'o-', color='darkred', linewidth=2, markersize=6)

# Custom spine
for spine_name in ['top', 'right', 'left', 'bottom']:
    ax.spines[spine_name].set_linewidth(1.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_position(('outward', 15))
ax.spines['bottom'].set_position(('outward', 10))

ax.set_title('Polished spines – clean, offset, no top/right')
ax.set_ylabel('Precipitation (mm/month)')
ax.set_xlabel('Month')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()



---
## Exercises – Module 07

1. **Axis limits**: Create a line plot of **Mekelle** rainfall from 2015–2020 only, with
   y-limits set to show only the 0–300 mm range.

2. **Tick locators**: For the monthly climatology, set x-axis ticks at every 2nd month and
   y-axis major ticks every 25 mm with minor ticks every 5 mm.

3. **Tick formatters**: Create a bar chart where y-tick labels show "XX mm" (with units)
   using `FormatStrFormatter`.

4. **Logarithmic scale**: Create a histogram of all CHIRPS rainfall data (all grid cells,
   one time step) using a log y-scale. What shape does the distribution have?

5. **Secondary axis**: Plot the Addis Ababa climatology with mm on the left and inches on the
   right using `ax.twinx()` and the `secondary_yaxis` functions approach.

6. **Aspect ratio**: Create a `pcolormesh` map of CHIRPS rainfall for January 2021 over
   Ethiopia with the correct latitude-adjusted aspect ratio.

7. **Spines**: Reproduce the "polished spines" example for the Addis Ababa time series
   (2000–2010) — hide top/right, offset left/bottom, colour the bottom spine blue.

### Mini-Project: Professional Axes for a Spatial-Temporal Analysis

Create a 2×2 figure that demonstrates different axes customisations:
- **Top-left**: Time series for Addis Ababa (1981–2022) with:
  - X-axis: YearLocator + YearFormatter
  - Y-axis: MultipleLocator(50), minor ticks every 10 mm
  - Spines: hide top/right, offset left/bottom by 10 pt
- **Top-right**: Bar chart of monthly climatology with:
  - Secondary y-axis in inches on the right
  - FuncFormatter on y-axis that adds "mm" suffix
- **Bottom-left**: Scatter of Gondar vs Addis Ababa monthly rainfall with:
  - aspect='equal'
  - x-limits and y-limits set to the same range
  - 1:1 reference line
- **Bottom-right**: pcolormap of January 2022 rainfall with:
  - Correct aspect ratio
  - Colourbar with formatted tick labels



In [ ]:
# Mini-Project starter
from matplotlib.dates import YearLocator, YearFormatter

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Top-left: Time series
ax = axes[0,0]
ax.plot(ts_df.index, ts_df['precip'], linewidth=0.5, color='steelblue')
ax.xaxis.set_major_locator(YearLocator(5))
ax.xaxis.set_major_formatter(YearFormatter())
ax.yaxis.set_major_locator(MultipleLocator(50))
ax.yaxis.set_minor_locator(MultipleLocator(10))
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.spines['left'].set_position(('outward', 10)); ax.spines['bottom'].set_position(('outward', 10))
ax.set_title('Time Series (1981–2022)'); ax.set_ylabel('mm/month')
ax.grid(True, which='both', alpha=0.2)

# Top-right: Climatology with secondary axis
ax = axes[0,1]
ax.bar(months, monthly_mean, color='steelblue', alpha=0.7)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:.0f} mm'))
ax2 = ax.secondary_yaxis('right', functions=(lambda x: x/25.4, lambda x: x*25.4))
ax2.set_ylabel('Inches')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.set_title('Climatology with Dual Units'); ax.tick_params(axis='x', rotation=45)

# Bottom-left: Gondar vs Addis scatter
gondar = ds.precip.isel(latitude=252, longitude=150).values
addis  = ds.precip.isel(latitude=180, longitude=174).values
ax = axes[1,0]
ax.scatter(gondar, addis, s=4, alpha=0.3, color='darkgreen')
ax.set_aspect('equal')
lim = [0, max(gondar.max(), addis.max())]
ax.set_xlim(lim); ax.set_ylim(lim)
ax.plot(lim, lim, 'r--', linewidth=0.8)
ax.set_xlabel('Gondar (mm/month)'); ax.set_ylabel('Addis Ababa (mm/month)')
ax.set_title('Gondar vs Addis Ababa')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Bottom-right: Spatial map
jan22 = ds.precip.isel(time=ds.time.dt.year == 2022, time=ds.time.dt.month == 1).squeeze()
ax = axes[1,1]
im = ax.pcolormesh(ds.longitude, ds.latitude, jan22, cmap='Blues', shading='auto')
ax.set_aspect(1.0 / np.cos(np.deg2rad(9)))
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('CHIRPS – January 2022')
cbar = plt.colorbar(im, ax=ax, format=FuncFormatter(lambda v, _: f'{v:.0f} mm'))
cbar.set_label('Precipitation (mm)')

plt.tight_layout()
plt.show()

